In [1]:
%pip install -U langchain langchain-core langchain-community rapidfuzz langchain-experimental langchain-text-splitters langchain-neo4j langchain-ollama neo4j python-dotenv

  Using cached langchain-1.3.12-py3-none-any.whl.metadata (6.0 kB)
  Using cached langchain_core-1.4.9-py3-none-any.whl.metadata (4.7 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
Using cached langchain-1.3.12-py3-none-any.whl (136 kB)
Using cached langchain_core-1.4.9-py3-none-any.whl (558 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.8
    Uninstalling langchain-core-1.4.8:
      Successfully uninstalled langchain-core-1.4.8
  Attempting uninstall: langchain-community━━━━━ 0/3 [langchain-core]
    Found existing installation: langchain-community 0.43 [langchain-core]
    Uninstalling langchain-community-0.4:━━━ 0/3 [langchain-core]
      Successfully uninstalled langchain-community-0.40/3 [langchain-core]
  Attempting uninstall: langchain0m━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [langchain-community]
    Found existing installation: langchain 1.3.

In [14]:
from dotenv import load_dotenv
import os

from pydantic import BaseModel, Field
from neo4j import GraphDatabase, Driver

from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_neo4j import Neo4jGraph, Neo4jVector
from langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars

from langchain_experimental.graph_transformers import LLMGraphTransformer

from langchain_community.graphs.graph_document import GraphDocument
from langchain_core.documents import Document
from langchain_community.graphs.graph_document import Node, Relationship

In [5]:
load_dotenv()

True

In [6]:
graph = Neo4jGraph(database="local")

def clean_graph():
    query = """
    MATCH (n)
    DETACH DELETE n
    """
    graph.query(query)

In [15]:
loader = TextLoader(file_path="../data/chapters/chapter_1.txt")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
documents = text_splitter.split_documents(documents=docs)

# for i in range(len(documents)): 
#    documents[i].metadata["chunk_id"] = f"chunk_{i+1}"
#    documents[i].metadata["chapter_id"] = "chapter_1"
#    documents[i].metadata["source"] = "../data/chapters/chapter_1.txt"

In [19]:
print(documents[:4])

[Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='I. A SCANDAL IN BOHEMIA\n\n\nI.'), Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='To Sherlock Holmes she is always _the_ woman. I have seldom heard him\nmention her under any other name. In his eyes she eclipses and\npredominates the whole of her sex. It was not that he felt any emotion\nakin to love for Irene Adler. All emotions, and that one particularly,\nwere abhorrent to his cold, precise but admirably balanced mind. He\nwas, I take it, the most perfect reasoning and observing machine that\nthe world has seen, but as a lover he would have placed himself in a\nfalse position. He never spoke of the softer passions, save with a gibe\nand a sneer. They were admirable things for the observer—excellent for\ndrawing the veil from men’s motives and actions. But for the trained\nreasoner to admit such intrusions into his own delicate and finely\nadjusted temperament was to introduc

In [339]:
import requests

print(requests.get("http://192.168.178.66:11434/api/tags", timeout=5).json())

{'models': [{'name': 'mistral:latest', 'model': 'mistral:latest', 'modified_at': '2026-06-16T20:09:42.9221651+02:00', 'size': 4372824384, 'digest': '6577803aa9a036369e481d648a2baebb381ebc6e897f2bb9a766a2aa7bfbc1cf', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '7.2B', 'quantization_level': 'Q4_K_M', 'context_length': 32768, 'embedding_length': 4096}, 'capabilities': ['completion', 'tools']}, {'name': 'llama3.1:latest', 'model': 'llama3.1:latest', 'modified_at': '2026-06-16T19:07:50.8696457+02:00', 'size': 4920753328, 'digest': '46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '8.0B', 'quantization_level': 'Q4_K_M', 'context_length': 131072, 'embedding_length': 4096}, 'capabilities': ['completion', 'tools']}, {'name': 'llama3.1:70b', 'model': 'llama3.1:70b', 'modified_at': '2026-06-16T17:11:25.965

In [16]:
llm = ChatOllama(
    model="qwen3.6:latest",
    base_url="http://192.168.178.67:11434",
    temperature=0,
    reasoning=False,
    keep_alive="24h",
    format="json",
    num_ctx=32768,       # context window: how much input+output the model can "see"
    num_predict=-1,     # output token limit: -1 = no limit (generate until done)
)
llm.invoke("Say hello in one sentence.")


# llm_transformer = LLMGraphTransformer(llm=llm)

# graph_documents = llm_transformer.convert_to_graph_documents(documents[:10])

AIMessage(content='{"answer": "Hello!"}', additional_kwargs={}, response_metadata={'model': 'qwen3.6:latest', 'created_at': '2026-07-09T19:52:22.837813Z', 'done': True, 'done_reason': 'stop', 'total_duration': 27471078000, 'load_duration': 26481104200, 'prompt_eval_count': 18, 'prompt_eval_duration': 587908000, 'eval_count': 8, 'eval_duration': 398303000, 'logprobs': None, 'model_name': 'qwen3.6:latest', 'model_provider': 'ollama'}, id='lc_run--019f486f-d0bc-7741-aa99-07465f08b087-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 8, 'total_tokens': 26})

In [17]:
system_prompt = """ You are a law and crime AI assistant with expertise in natural language processing, 
specifically in named entity recognition (NER) for cases, persons, relationships, places and events. 

Your task is to analyze the given text and identify and extract all 
case-related entities and relationships from the given text.

output: For each document return a GraphDocument Object containing nodes, relationships and source= (the provided source document)

nodes: id, type (e.g. Person), properties (dict) containing description (the name of the node), context and further information such as traits or physical description.

relationship: source (node id), target (node id), type (e.g. WORKS_FOR), properties: dict with further information such as context. 

"""

# You must generate the output in a JSON format containing a list with JSON objects.

# system_message = SystemMessage(content=system_prompt)

# user_prompt = """
# Task: Analyse the following text and extract all relevant entities and relationships. 



# examples: JSON Objects following this schema: 

# """

# human_prompt = HumanMessagePromptTemplate(
#     prompt=PromptTemplate(template=user_prompt)
# )

# chat_prompt = ChatPromptTemplate.from_messages([system_message, human_prompt])

# human_prompt = """
# Analyze the following text and extract nodes and relationships follwing this pattern: 
# nodes: id:int, type: str (i.e. Person), properties: object (may contain e.g. context or a description (the name of the node))  
# relationships: source: Node, target: Node, type: str (i.e. WORKS_FOR), properties: Object. 

# For the nodes, extract also the source metadata (chunk_id, chapter_id, source).

# RETURN ONLY THE EXTRACTED JSON OBJECTS
# """

prompt = """
You are a top-tier algorithm designed for extracting information in structured formats to build a knowledge graph. Your task is to identify the entities and relations specified in the user prompt from a given text and produce the output in JSON format. This output should be a list of JSON objects, with each object containing the following keys:

- **"id"**: The name of the extracted entity,.
- **"type"**: The type of the extracted entity
- **"properties"**: A nested JSON Object containing further attributes that describe the node
- **"source"**: The text of the entity representing the tail of the relation.
- **"tail_type"**: The type of the tail entity, also selected from the provided list of types.

Extract as many entities and relationships as possible.

**Entity Consistency**: Ensure consistency in entity representation. If an entity, like "John Doe," appears multiple times in the text under different names or pronouns (e.g., "Joe," "he"), use the most complete identifier consistently. This consistency is essential for creating a coherent and easily understandable knowledge graph.

**Important Notes**:
- Do not add any extra explanations or text.
"""

prompt_2 = """
# Role: You are a law and crime AI assistant with expertise in natural language processing, 
specifically in named entity recognition (NER) for cases, persons, relationships, places and events. 

# Task: Your task is to analyze the given text and identify and extract all 
case-related entities and relationships from the given text.

# Input: An array of document object containing metadata and the chunked text

# Output: A List Of GraphDocument Objects.

IMPORTANT!! GENERATE ONE GraphDocument FOR EVERY Document. ALSO INCLUDE THE SOURCE STORED IN THE DOCUMENTS METADATA

# Maintain Entity Consistency: When extracting entities, it's vital to ensure consistency.
If an entity, such as "Sherlock Holmes", is mentioned multiple times in the text
but is referred to by different names or pronouns (e.g., "Holmes", "he"),
always use the most complete identifier for that entity throughout the
knowledge graph.

# Instructions: Every GraphDocument should only contain the nodes and relationships extracted from the page_content of the Document listed in the GraphDocuments Source!!

Here is an example output: 

[GraphDocument(nodes=[], relationships=[], source=Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='I. A SCANDAL IN BOHEMIA\n\n\nI.')),
 GraphDocument(nodes=[Node(id='Irene Adler', type='Person', properties={}), Node(id='Sherlock Holmes', type='Person', properties={})], relationships=[Relationship(source=Node(id='Sherlock Holmes', type='Person', properties={}), target=Node(id='Irene Adler', type='Person', properties={}), type='IS_ADMIRER_OF', properties={})], source=Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='To Sherlock Holmes she is always _the_ woman. I have seldom heard him\nmention her under any other name. In his eyes she eclipses and\npredominates the whole of her sex. It was not that he felt any emotion\nakin to love for Irene Adler. All emotions, and that one particularly,\nwere abhorrent to his cold, precise but admirably balanced mind. He\nwas, I take it, the most perfect reasoning and observing machine that\nthe world has seen, but as a lover he would have placed himself in a\nfalse position. He never spoke of the softer passions, save with a gibe\nand a sneer. They were admirable things for the observer—excellent for\ndrawing the veil from men’s motives and actions. But for the trained\nreasoner to admit such intrusions into his own delicate and finely\nadjusted temperament was to introduce a distracting factor which might\nthrow a doubt upon all his mental results. Grit in a sensitive\ninstrument, or a crack in one of his own high-power lenses, would not')),
 GraphDocument(nodes=[Node(id='Irene Adler', type='Person', properties={}), Node(id='his', type='Person', properties={})], relationships=[Relationship(source=Node(id='Irene Adler', type='Person', properties={}), target=Node(id='his', type='Person', properties={}), type='IS_LOVED_BY', properties={})], source=Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='throw a doubt upon all his mental results. Grit in a sensitive\ninstrument, or a crack in one of his own high-power lenses, would not\nbe more disturbing than a strong emotion in a nature such as his. And\nyet there was but one woman to him, and that woman was the late Irene\nAdler, of dubious and questionable memory.')),
 GraphDocument(nodes=[Node(id='Odessa', type='Location', properties={}), Node(id='Holmes', type='Person', properties={}), Node(id='official police', type='Organisation', properties={}), Node(id='Baker Street', type='Location', properties={})], relationships=[Relationship(source=Node(id='Holmes', type='Person', properties={}), target=Node(id='Baker Street', type='Location', properties={}), type='RESIDES_IN', properties={}), Relationship(source=Node(id='Holmes', type='Person', properties={}), target=Node(id='Odessa', type='Location', properties={}), type='SUMMONED_TO', properties={}), Relationship(source=Node(id='Holmes', type='Person', properties={}), target=Node(id='official police', type='Organisation', properties={}), type='WORKS_FOR', properties={})], source=Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='I had seen little of Holmes lately. My marriage had drifted us away\nfrom each other. My own complete happiness, and the home-centred\ninterests which rise up around the man who first finds himself master\nof his own establishment, were sufficient to absorb all my attention,\nwhile Holmes, who loathed every form of society with his whole Bohemian\nsoul, remained in our lodgings in Baker Street, buried among his old\nbooks, and alternating from week to week between cocaine and ambition,\nthe drowsiness of the drug, and the fierce energy of his own keen\nnature. He was still, as ever, deeply attracted by the study of crime,\nand occupied his immense faculties and extraordinary powers of\nobservation in following out those clues, and clearing up those\nmysteries which had been abandoned as hopeless by the official police.\nFrom time to time I heard some vague account of his doings: of his\nsummons to Odessa in the case of the Trepoff murder, of his clearing up')),
 GraphDocument(nodes=[Node(id='Odessa', type='Location', properties={}), Node(id='reigning family of Holland', type='Organisation', properties={}), Node(id='Trincomalee', type='Location', properties={}), Node(id='Holland', type='Location', properties={})], relationships=[Relationship(source=Node(id='Holland', type='Location', properties={}), target=Node(id='reigning family of Holland', type='Organisation', properties={}), type='GOVERNED_BY', properties={})], source=Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='From time to time I heard some vague account of his doings: of his\nsummons to Odessa in the case of the Trepoff murder, of his clearing up\nof the singular tragedy of the Atkinson brothers at Trincomalee, and\nfinally of the mission which he had accomplished so delicately and\nsuccessfully for the reigning family of Holland. Beyond these signs of\nhis activity, however, which I merely shared with all the readers of\nthe daily press, I knew little of my former friend and companion.'))]

"""
temp = ChatPromptTemplate.from_messages

extract_prompt = """
You are a high-precision information extraction system designed to construct knowledge graphs from text.
Your task is to extract entities and relationships from the given text and return them in a structured format as JSON objects.
Follow these rules strictly:

# General Principles

- Extract as much relevant information as possible without inventing or inferring anything not explicitly stated.
- Prioritize accuracy over completeness.
- Output only structured data. Do not include explanations, reasoning, or commentary.

# Nodes (id, type, properties)

- Nodes represent entities or concepts.
- Use simple, generic labels (e.g., "Person", "Location", "Organisation").
- Do not use overly specific types (e.g., avoid "Detective", use "Person").
- Node IDs must be human-readable names exactly as found in the text.
- Never use numeric IDs.
- Use a dict called properties to further describe the nodes. If there are no desciptions in the text, return an empty dict.

# Relationships (source, target, type, properties)

- Relationships represent connections between nodes.
- source and target both have to be node objects in the same format as described above.
- Use general, timeless relationship types (e.g., "RESIDES_IN", "ASSISTS", "KNOWS").
- Avoid context-specific or time-dependent labels (e.g., do NOT use "BECAME_PROFESSOR").
- Ensure consistency across all relationships.
- Use a dict called properties to further describe the relationships. If there are no desciptions in the text, return an empty dict.

# Document (metadata, page_content)

- the metadata and page_content of the source document 

# Coreference Resolution

- Resolve all references to the same entity (e.g., pronouns or aliases).
- Always use the most complete and explicit name found in the text as the node ID.
- Ensure that each real-world entity appears only once in the graph.

# Output Format

IMPORTANT!! Return only valid JSON.
Do not return Python code.
Do not return GraphDocument(...).
Do not include explanations.
Do not use markdown fences.

JSON structure: 
- nodes: list of Node(id, type, properties=\\{\\})
- relationships: list of Relationship(source, target, type, properties={})
- document: metadata={...}, page_content='...'

ESSENTIAL: ALWAYS INCLUDE THE SOURCE DOCUMENTs PAGE_CONTENT AND METADATA IN THE DOCUMENT OBJECT!!!!!

example: {'nodes': [{'id': 'Sherlock Holmes', 'type': 'Person', 'properties': {}},
   {'id': 'Irene Adler', 'type': 'Person', 'properties': {}}],
  'relationships': [{'source': {'id': 'Sherlock Holmes',
     'type': 'Person',
     'properties': {}},
    'target': {'id': 'Irene Adler', 'type': 'Person', 'properties': {}},
    'type': 'KNOWS',
    'properties': {}}],
  'source': {'metadata': {'source': '../data/chapters/chapter_1.txt'},
   'page_content': 'I. A SCANDAL IN BOHEMIA\n\nI.\n\n'}}

IMPORTANT!!
**Do not include any text outside this structure.**
**Do not include explanations or intermediate steps.**


Input document: 
"""

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_experimental.graph_transformers import LLMGraphTransformer

# The system message is YOUR instructions only.
# LLMGraphTransformer appends its own JSON schema requirements automatically.
# {input} in the human message is MANDATORY — do not remove it.
system_prompt = (
    "You are a law and crime AI assistant specializing in named entity recognition (NER) "
    "for literary detective fiction. "
    "Extract case-related entities and relationships: persons, locations, organisations, "
    "objects, events, crimes, and documents. "
    "Only extract entities and relationships explicitly stated in the text. "
    "Use full names where possible (e.g. 'Sherlock Holmes', not 'Holmes'). "
    "# IMPORTANT: Only use these node types: "
    "Person, Location, Organisation, Event, Object, Document, Crime, Disguise, Role. "
    "Only use these relationship types: "
    "KNOWS, EMPLOYS, MARRIED_TO, LIVES_AT, LOCATED_IN, OWNS, SEEKS, WROTE, "
    "USES_DISGUISE, INVOLVED_IN, HAS_ROLE, THREATENS, VISITED, MEMBER_OF, "
    "RESIDES_IN, SUMMONED_TO, GOVERNED_BY, IS_ADMIRER_OF."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", (
        "Tip: Make sure to answer in the correct format and do not include any explanations. "
        "Use the given format to extract information from the following input: {input}"
    )),
])

merge_prompt = """You are an entity resolution system for knowledge graphs.
Your task is to identify and merge nodes that refer to the same real-world entity within a graph. This includes resolving pronouns, aliases, partial names, and descriptive references.
Think step by step and Follow these rules strictly:

# Core Objective

- Detect nodes that refer to the same entity (e.g., "he", "Holmes", "Sherlock Holmes" or "Watson" and "Dr. Watson").
- Replace the incomplete description with the canonical one (e.g. replace he with Sherlock Holmes)
- Make sure to also update the relationships!!!
- Do not merge entities unless there is clear textual evidence.
- Canonical Naming
- Always choose the most complete and explicit name as the final node ID.
- Example: use "Sherlock Holmes" instead of "Holmes" or "he".
- Example: use "Irene Adler" instead of "the woman".

# Merge Rules

- Pronouns (he, she, they, him, her, you) must be resolved to the correct entity.
- Titles or descriptions (e.g., "the woman", "the detective") should be mapped to named entities when clearly identifiable.
- Partial names (e.g., "Holmes" or "Watson") must be merged with full names when unambiguous.
- Do not merge entities with ambiguous references.

# Graph Update

- Replace all occurrences of duplicate or resolved nodes with the canonical node.
- Update all relationships to point to the canonical node.
- Remove duplicate nodes after merging.
- Preserve all relationships and do not lose information.

# Output Format

- Return the updated JSON Objects with merged nodes.
- Maintain the original structure!!! Don't create new JSON OBjects! Only modify the objects inside!

nodes
relationships
document

Do not include explanations or reasoning. Output only the final graph.

# Output Format

IMPORTANT!! Return only valid JSON.
Do not return Python code.
Do not return GraphDocument(...).
Do not include explanations.
Do not use markdown fences.

JSON structure: 
- nodes: list of Node(id, type, properties=\\{\\})
- relationships: list of Relationship(source, target, type, properties={})
- document: metadata={...}, page_content='...'

ESSENTIAL: ALWAYS INCLUDE THE SOURCE DOCUMENTs PAGE_CONTENT AND METADATA IN THE DOCUMENT OBJECT!!!!!

example: {'nodes': [{'id': 'Sherlock Holmes', 'type': 'Person', 'properties': {}},
   {'id': 'Irene Adler', 'type': 'Person', 'properties': {}}],
  'relationships': [{'source': {'id': 'Sherlock Holmes',
     'type': 'Person',
     'properties': {}},
    'target': {'id': 'Irene Adler', 'type': 'Person', 'properties': {}},
    'type': 'KNOWS',
    'properties': {}}],
  'source': {'metadata': {'source': '../data/chapters/chapter_1.txt'},
   'page_content': 'I. A SCANDAL IN BOHEMIA\n\nI.\n\n'}}

IMPORTANT!!
**Do not include any text outside this structure.**
**Do not include explanations or intermediate steps.**


Input document: 
"""

In [19]:
from copy import deepcopy
import os
import json


# Merge Nodes
all_nodes = []
all_relationships = []
next_free_id = 1
id_map = {}


def merge_nodes(nodes:any):
    for node in nodes:
        curr_node = next(
            (
                n
                for n in all_nodes
                if n["type"] == node["type"]
                and n["properties"]["description"] == node["properties"]["description"]
            ),
            None,
        )

        if curr_node is not None:
            curr_props = curr_node["properties"]
            new_props = node["properties"]
            new_keys = new_props.keys()

            for key in new_keys:
                if key in curr_props:
                    if isinstance(new_props[key], str):
                        if curr_props[key] != new_props[key]:
                            attributes = curr_props[key].split(',') + new_props[key].split(',')
                            curr_props[key] = ', '.join(dict.fromkeys(attributes))
                            # curr_props[key] = ' '.join(dict.fromkeys((', '.join([curr_props[key], new_props[key]])).split()))
                            
                        pass
                    else:
                        curr_props[key] = list(set(curr_props[key] + new_props[key]))
                else:
                    curr_props[key] = new_props[key]
            
            id_map[node["id"]] = curr_node


            
        else:
            canonical_node = deepcopy(node)
            canonical_node["id"] = next_free_id
            all_nodes.append(canonical_node)
            id_map[node["id"]] = canonical_node
            next_free_id+=1

    print(f"nodes: {len(all_nodes)}")
    return all_nodes

def merge_relationships(relationships:any):

    for relationship in relationships:
        
        canonical_source_node = id_map.get(relationship["source"])
        canonical_target_node = id_map.get(relationship["target"])
        curr_rel = next(
            (
                r
                for r in all_relationships
                if r["type"] == relationship["type"]
                and r["source"] == canonical_source_node["id"] and r["target"] == canonical_target_node["id"]
            ),
            None,
        )

        if curr_rel is not None:
            curr_props = curr_rel["properties"]
            new_props = relationship["properties"]
            new_keys = new_props.keys()

            for key in new_keys:
                if key in curr_props:
                    if isinstance(new_props[key], str):
                        if curr_props[key] != new_props[key]:
                            attributes = curr_props[key].split(',') + new_props[key].split(',')
                            curr_props[key] = ', '.join(dict.fromkeys(attributes))
                            # curr_props[key] = ' '.join(dict.fromkeys((', '.join([curr_props[key], new_props[key]])).split()))
                            
                        pass
                    else:
                        curr_props[key] += new_props[key]
                else:
                    curr_props[key] = new_props[key]

        else:
            canonical_relationship = deepcopy(relationship)
            canonical_relationship["source"] = canonical_source_node["id"]
            canonical_relationship["target"] = canonical_target_node["id"]
            all_relationships.append(canonical_relationship)

    print(f"relationships: {len(all_relationships)}")    
    return all_relationships
            


In [20]:
from langchain_core.documents import Document

text = """
To Sherlock Holmes she is always _the_ woman. I have seldom heard him
mention her under any other name. In his eyes she eclipses and
predominates the whole of her sex. It was not that he felt any emotion
akin to love for Irene Adler. All emotions, and that one particularly,
were abhorrent to his cold, precise but admirably balanced mind. He
was, I take it, the most perfect reasoning and observing machine that
the world has seen, but as a lover he would have placed himself in a
false position. He never spoke of the softer passions, save with a gibe
and a sneer. They were admirable things for the observer—excellent for
drawing the veil from men’s motives and actions. But for the trained
reasoner to admit such intrusions into his own delicate and finely
adjusted temperament was to introduce a distracting factor which might
throw a doubt upon all his mental results. Grit in a sensitive
instrument, or a crack in one of his own high-power lenses, would not
be more disturbing than a strong emotion in a nature such as his. And
yet there was but one woman to him, and that woman was the late Irene
Adler, of dubious and questionable memory.
"""

text_documents = [Document(page_content=text)]

allowed_nodes = [
    "Person",
    "Location",
    "Organisation",
    "Role",
    "Object",
    "Document",
    "Case",
    "Event",
    "Date",
    "Disguise",
    "Crime",
    "Alias"
]
allowed_relationships = [
    "KNOWS",
    "EMPLOYS",
    "MARRIED_TO",
    "LVES_AT",
    "LOCATED_IN",
    "OWNS",
    "SEEKS",
    "WROTE",
    "USES_DISGUISE",
    "USES_ALIAS",
    "INVOLVED_IN",
    "HAS_ROLE",
    "THREATENS",
    "VISITED",
    "MEMBER_OF"
]
# allow_rels = [
#     ("Person", "ADMIRES", "Person"),
#     ("Person", "RESIDES_AT", "Location"),
#     ("Person", "WORKS_FOR", "Person"),
#     ("Person", "COMES_TO", "Person"),
#     ("Person", "WITNESSED", "Event"),
#     ("Person", "WROTE", "Document"),
#     ("Person", "POSSESSES", "Document"),
#     ("Person", "POSSESSES", "Object"),
#     ("Event", "TOOK_PLACE_AT", "Location"),
#     ("Event", "TOOK_PLACE_AT", "Date"),
# ]

# no_schema = LLMGraphTransformer(
#     llm=llm,
#     ignore_tool_usage=True,
#     # allowed_nodes=allowed_nodes,
#     # node_properties=True,
#     # relationship_properties=True,
#     # allowed_relationships=allowed_relationships,
#     # strict_mode=False,
#     # prompt=system_prompt
#     prompt=prompt
# )

# data = await no_schema.aconvert_to_graph_documents(documents=documents[:10])
len(text)

1149

In [21]:
def serialize_node(node):
    return Node(id=node["id"], type=node["type"], properties=node["properties"])


def serialize_relationship(relation, nodes: list[Node]):
    source = Node(id=relation["source"]["id"], type=relation["source"]["type"], properties=relation["source"]["properties"])
    target = Node(id=relation["target"]["id"], type=relation["target"]["type"], properties=relation["target"]["properties"])
    return Relationship(
        source=source,
        target=target,
        type=relation["type"],
        properties=relation["properties"],
    )

def cast_to_graph_document(json_object:any):
    nodes: list[Node] = []
    for node in json_object["nodes"]:
        nodes.append(serialize_node(node))

    relationships: list[Relationship] = []
    for rel in json_object["relationships"]:
        relationships.append(serialize_relationship(rel, nodes))
    
    source: Document = Document(metadata=json_object["document"]["metadata"], page_content=json_object["document"]["page_content"])
    return GraphDocument(nodes=nodes, relationships=relationships, source=source)


def convert_to_graph_doc(s: str):
    graph_documents = eval(s, {
        "GraphDocument": GraphDocument,
        "Document": Document,
        "Node": Node,
        "Relationship": Relationship,
    })
    return graph_documents

def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

In [12]:
import json

print("# Extraction: \n")
test_data = documents[:5]

all_results= []
for i in range(len(test_data)):
    response = llm.invoke(f"system_prompt: {extract_prompt} {test_data}")
    json_response = json.loads(response.content)
    # graph_doc = cast_to_graph_document(json_response)
    all_results.append(json_response)
    print(f"extracted {len(all_results)}/{len(test_data)}")


# Extraction: 

extracted 1/5
extracted 2/5
extracted 3/5
extracted 4/5
extracted 5/5


In [ ]:
print("# Merging: \n")

all_merge_results = []
for chunk in all_results:
    merged_results = llm.invoke(f"system_prompt: {merge_prompt} {chunk}")
    all_merge_results.append(json.loads(merged_results.content))
    print(f"merged {len(all_merge_results)}/{len(all_results)}")

JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 3 (char 2)

In [ ]:
print("# Cleaning: \n")
clean_mer_res = []
for res in all_merge_results:
    clean_mer_res.extend(convert_to_graph_doc(res))
    print(f"cleaned {len(clean_mer_res)}/{len(all_merge_results)}")

In [ ]:
def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

all_merge_results = []

for batch in chunk_list(all_results, 5):
    merged_results = llm.invoke(f"system_prompt: {merge_prompt} {batch}")
    all_merge_results.append(merged_results.content)


In [199]:
all_results

[{'nodes': [{'id': 'Sherlock Holmes', 'type': 'Person', 'properties': {}},
   {'id': 'Irene Adler', 'type': 'Person', 'properties': {}}],
  'relationships': [{'source': {'id': 'Sherlock Holmes',
     'type': 'Person',
     'properties': {}},
    'target': {'id': 'Irene Adler', 'type': 'Person', 'properties': {}},
    'type': 'KNOWS',
    'properties': {}}],
  'document': {'metadata': {'source': '../data/chapters/chapter_1.txt'},
   'page_content': 'I. A SCANDAL IN BOHEMIA\n\n\nI.\n\nTo Sherlock Holmes she is always _the_ woman. I have seldom heard him\nmention her under any other name. In his eyes she eclipses and\npredominates the whole of her sex. It was not that he felt any emotion\nakin to love for Irene Adler. All emotions, and that one particularly,\nwere abhorrent to his cold, precise but admirably balanced mind. He\nwas, I take it, the most perfect reasoning and observing machine that\nthe world has seen, but as a lover he would have placed himself in a\nfalse position. He neve

In [189]:
# with open("../data/local/chapter_1.txt", "w") as file_out:
#     file_out.write(str(all_results))
len(all_results)

28

In [178]:
clean_graph()
graph.add_graph_documents(
    all_results,
    include_source=True,
    baseEntityLabel=True
)

In [201]:
all_merge_results

["[GraphDocument(nodes=[Node(id='Sherlock Holmes', type='Person', properties={}), Node(id='Irene Adler', type='Person', properties={}), Node(id='Baker Street', type='Location', properties={}), Node(id='Odessa', type='Location', properties={}), Node(id='Trepoff', type='Person', properties={}), Node(id='Atkinson brothers', type='Person', properties={}), Node(id='Trincomalee', type='Location', properties={}), Node(id='Holland', type='Location', properties={}), Node(id='Dr. Watson', type='Person', properties={}), Node(id='Mary Jane', type='Person', properties={})], relationships=[Relationship(source=Node(id='Sherlock Holmes', type='Person', properties={}), target=Node(id='Irene Adler', type='Person', properties={}), type='KNOWS', properties={}), Relationship(source=Node(id='Sherlock Holmes', type='Person', properties={}), target=Node(id='Baker Street', type='Location', properties={}), type='RESIDES_IN', properties={}), Relationship(source=Node(id='Sherlock Holmes', type='Person', propertie

In [207]:
clean_mer_res = []
for res in all_merge_results:
    clean_mer_res.extend(convert_to_graph_doc(res))


In [208]:
clean_mer_res

[GraphDocument(nodes=[Node(id='Sherlock Holmes', type='Person', properties={}), Node(id='Irene Adler', type='Person', properties={}), Node(id='Baker Street', type='Location', properties={}), Node(id='Odessa', type='Location', properties={}), Node(id='Trepoff', type='Person', properties={}), Node(id='Atkinson brothers', type='Person', properties={}), Node(id='Trincomalee', type='Location', properties={}), Node(id='Holland', type='Location', properties={}), Node(id='Dr. Watson', type='Person', properties={}), Node(id='Mary Jane', type='Person', properties={})], relationships=[Relationship(source=Node(id='Sherlock Holmes', type='Person', properties={}), target=Node(id='Irene Adler', type='Person', properties={}), type='KNOWS', properties={}), Relationship(source=Node(id='Sherlock Holmes', type='Person', properties={}), target=Node(id='Baker Street', type='Location', properties={}), type='RESIDES_IN', properties={}), Relationship(source=Node(id='Sherlock Holmes', type='Person', properties=

In [209]:
clean_graph()
graph.add_graph_documents(
    clean_mer_res,
    include_source=True,
    baseEntityLabel=True
)

In [ ]:

def convert_to_graph_doc(s: str):
    graph_documents = eval(s, {
        "GraphDocument": GraphDocument,
        "Document": Document,
        "Node": Node,
        "Relationship": Relationship,
    })
    return graph_documents


GraphDocument(nodes=[Node(id='Sherlock Holmes', type='Person', properties={}), Node(id='Irene Adler', type='Person', properties={})], relationships=[Relationship(source=Node(id='Sherlock Holmes', type='Person', properties={}), target=Node(id='Irene Adler', type='Person', properties={}), type='KNOWS', properties={})], source=Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='I. A SCANDAL IN BOHEMIA\n\n\nI.\n\nTo Sherlock Holmes she is always _the_ woman. I have seldom heard him\nmention her under any other name. In his eyes she eclipses and\npredominates the whole of her sex. It was not that he felt any emotion\nakin to love for Irene Adler. All emotions, and that one particularly,\nwere abhorrent to his cold, precise but admirably balanced mind. He\nwas, I take it, the most perfect reasoning and observing machine that\nthe world has seen, but as a lover he would have placed himself in a\nfalse position. He never spoke of the softer passions, save with a gibe\

In [181]:
convert_to_graph_doc(merged_results.content)

SyntaxError: '(' was never closed (<string>, line 1)

In [166]:
clean_graph()
graph.add_graph_documents(
    convert_to_graph_doc(merged_results.content),
    baseEntityLabel=True,
    include_source=True,
)

In [98]:

def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

all_graph_docs = []

for batch in chunk_list(documents[10:20], 4):
    batch_results = await llm.ainvoke(f"system_prompt: {test_prompt} {batch}")
    # merged_batch_results = await llm.ainvoke(f"system_prompt: {merge_prompt} {convert_to_graph_doc(batch_results.content)}")
    all_graph_docs.extend(convert_to_graph_doc(batch_results.content))
    print(f"Processed {len(all_graph_docs)}/{len(documents)} documents")

SyntaxError: closing parenthesis ']' does not match opening parenthesis '(' (<string>, line 1)

In [89]:
all_graph_docs

[GraphDocument(nodes=[Node(id='Sherlock Holmes', type='Person', properties={'traits': ['cold', 'precise', 'admirably balanced mind', 'perfect reasoning and observing machine', 'Bohemian soul', 'keen nature']}), Node(id='Irene Adler', type='Person', properties={'status': 'late'}), Node(id='Baker Street', type='Location', properties={})], relationships=[Relationship(source=Node(id='Sherlock Holmes', type='Person', properties={'traits': ['cold', 'precise', 'admirably balanced mind', 'perfect reasoning and observing machine', 'Bohemian soul', 'keen nature']}), target=Node(id='Irene Adler', type='Person', properties={'status': 'late'}), type='IS_AFFECTED_BY', properties={'nature': 'emotional abhorrence'}), Relationship(source=Node(id='Sherlock Holmes', type='Person', properties={'traits': ['cold', 'precise', 'admirably balanced mind', 'perfect reasoning and observing machine', 'Bohemian soul', 'keen nature']}), target=Node(id='Irene Adler', type='Person', properties={'status': 'late'}), typ

In [90]:
merged_result = llm.invoke(f"system_prompt: {merge_prompt} {all_graph_docs}")

In [91]:
merged_graph_doc = convert_to_graph_doc(merged_result.content)

In [92]:
merged_graph_doc

[GraphDocument(nodes=[Node(id='Sherlock Holmes', type='Person', properties={'traits': ['cold', 'precise', 'admirably balanced mind', 'perfect reasoning and observing machine', 'Bohemian soul', 'keen nature']}), Node(id='Irene Adler', type='Person', properties={'status': 'late'}), Node(id='Baker Street', type='Location', properties={}), Node(id='Watson', type='Person', properties={}), Node(id='Trepoff', type='Person', properties={}), Node(id='Atkinson brothers', type='Person', properties={}), Node(id='Holland', type='Location', properties={}), Node(id='Mary Jane', type='Person', properties={})], relationships=[Relationship(source=Node(id='Sherlock Holmes', type='Person', properties={'traits': ['cold', 'precise', 'admirably balanced mind', 'perfect reasoning and observing machine', 'Bohemian soul', 'keen nature']}), target=Node(id='Irene Adler', type='Person', properties={'status': 'late'}), type='IS_AFFECTED_BY', properties={'nature': 'emotional abhorrence'}), Relationship(source=Node(i

In [69]:
clean_graph()

In [70]:
graph.add_graph_documents(
    all_graph_docs,
    baseEntityLabel=True,
    include_source=True,
)

In [36]:
from langchain_community.graphs.graph_document import GraphDocument
from langchain_core.documents import Document
from langchain_community.graphs.graph_document import Node, Relationship

print(len(str(response.content)))
response.content
graph_documents = eval(response.content, {
    "GraphDocument": GraphDocument,
    "Document": Document,
    "Node": Node,
    "Relationship": Relationship,
})
graph_documents

5022


[GraphDocument(nodes=[], relationships=[], source=Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='I. A SCANDAL IN BOHEMIA\n\n\nI.')),
 GraphDocument(nodes=[Node(id='Sherlock Holmes', type='Person', properties={}), Node(id='Irene Adler', type='Person', properties={})], relationships=[Relationship(source=Node(id='Sherlock Holmes', type='Person', properties={}), target=Node(id='Irene Adler', type='Person', properties={}), type='IS_ADMIRER_OF', properties={})], source=Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='To Sherlock Holmes she is always _the_ woman. I have seldom heard him\nmention her under any other name. In his eyes she eclipses and\npredominates the whole of her sex. It was not that he felt any emotion\nakin to love for Irene Adler. All emotions, and that one particularly,\nwere abhorrent to his cold, precise but admirably balanced mind. He\nwas, I take it, the most perfect reasoning and observing machine that\nthe 

In [28]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True,
)

Nodes: [Node(id='Irene Adler', type='Person', properties={}), Node(id='Sherlock Holmes', type='Person', properties={})]
Relationships: [Relationship(source=Node(id='Sherlock Holmes', type='Person', properties={}), target=Node(id='Irene Adler', type='Person', properties={}), type='KNOWS', properties={}), Relationship(source=Node(id='Sherlock Holmes', type='Person', properties={}), target=Node(id='Irene Adler', type='Person', properties={}), type='MENTIONS', properties={})]


In [81]:
data

[GraphDocument(nodes=[], relationships=[], source=Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='I. A SCANDAL IN BOHEMIA\n\n\nI.')),
 GraphDocument(nodes=[], relationships=[], source=Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='To Sherlock Holmes she is always _the_ woman. I have seldom heard him\nmention her under any other name. In his eyes she eclipses and\npredominates the whole of her sex. It was not that he felt any emotion\nakin to love for Irene Adler. All emotions, and that one particularly,\nwere abhorrent to his cold, precise but admirably balanced mind. He\nwas, I take it, the most perfect reasoning and observing machine that\nthe world has seen, but as a lover he would have placed himself in a\nfalse position. He never spoke of the softer passions, save with a gibe\nand a sneer. They were admirable things for the observer—excellent for\ndrawing the veil from men’s motives and actions. But for the trained\nreas

In [82]:
for d in data:
    print(d.nodes)

[]
[]
[]
[]
[]


In [28]:
llm_transformer = LLMGraphTransformer(
    llm=llm,
    prompt=system_prompt
#     allowed_nodes=["Person", "Location", "Organization", "Event", "Work", "Date", "Substance"],
#     allowed_relationships=[
#         "KNOWS",
#         "LIVES_IN",
#         "VISITS",
#         "WORKS_FOR",
#         "OWNS",
#         "INVOLVED_IN",
#         "INVESTIGATES",
#         "ASSOCIATED_WITH",
#         "USES",
#         "INFORMED_BY",
#         "SURPASSES",
#         "MENTIONED_IN",
#         "COMPANION_OF",
#         "ACCOMPLISHED"
#     ],
#     ignore_tool_usage=True,
#     strict_mode=False
)
graph_documents = llm_transformer.convert_to_graph_documents(documents[:10])



# prompt = ChatPromptTemplate.from_messages([("system", system_prompt), ("human", human_prompt)])

# prompt_result = llm.invoke(f"System prompt: {system_prompt}, user prompt: {user_prompt}, text: {documents}")

# chain = prompt | llm | StrOutputParser()

# response = chain.invoke()

# response = llm.invoke([("system", system_prompt), "human", human_prompt])

TypeError: Expected a Runnable, callable or dict.Instead got an unsupported type: <class 'str'>

In [20]:
graph_documents

[GraphDocument(nodes=[], relationships=[], source=Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='I. A SCANDAL IN BOHEMIA\n\n\nI.')),
 GraphDocument(nodes=[], relationships=[], source=Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='To Sherlock Holmes she is always _the_ woman. I have seldom heard him\nmention her under any other name. In his eyes she eclipses and\npredominates the whole of her sex. It was not that he felt any emotion')),
 GraphDocument(nodes=[], relationships=[], source=Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='akin to love for Irene Adler. All emotions, and that one particularly,\nwere abhorrent to his cold, precise but admirably balanced mind. He\nwas, I take it, the most perfect reasoning and observing machine that')),
 GraphDocument(nodes=[], relationships=[], source=Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='the world has seen, but as 

In [ ]:
{\n    "id": 1,\n    "type": "Person",\n    "properties": {"name": "Irene Adler"},\n    "source": {\n      "chunk_id": "chunk_3",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 2,\n    "type": "Person",\n    "properties": {"name": "Sherlock Holmes"},\n    "source": {\n      "chunk_id": "chunk_4",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 3,\n    "type": "Person",\n    "properties": {"name": "Irene Adler"},\n    "source": {\n      "chunk_id": "chunk_4",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 4,\n    "type": "Person",\n    "properties": {"name": "Sherlock Holmes"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 5,\n    "type": "Person",\n    "properties": {"name": "Sherlock Holmes"},\n    "source": {\n      "chunk_id": "chunk_2",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 6,\n    "type": "Person",\n    "properties": {"name": "Irene Adler"},\n    "source": {\n      "chunk_id": "chunk_2",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 7,\n    "type": "Event",\n    "properties": {"name": "The Trepoff murder"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 8,\n    "type": "Event",\n    "properties": {"name": "The Atkinson brothers at Trincomalee"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 9,\n    "type": "Event",\n    "properties": {"name": "The mission for the reigning family of Holland"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 10,\n    "type": "Place",\n    "properties": {"name": "Baker Street"},\n    "source": {\n      "chunk_id": "chunk_4",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 11,\n    "type": "Place",\n    "properties": {"name": "Odessa"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 12,\n    "type": "Place",\n    "properties": {"name": "Trincomalee"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 13,\n    "type": "Place",\n    "properties": {"name": "Holland"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 14,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_4",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 15,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 16,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 17,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 18,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 19,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 20,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 21,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 22,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 23,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 24,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 25,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 26,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 27,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 28,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 29,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 30,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 31,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 32,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 33,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 34,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 35,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 36,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 37,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 38,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 39,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 40,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 41,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 42,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 43,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 44,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 45,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 46,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 47,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 48,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 49,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 50,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 51,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 52,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 53,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 54,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 55,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 56,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 57,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 58,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 59,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 60,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 61,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 62,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 63,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 64,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 65,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 66,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 67,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 68,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 69,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 70,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 71,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 72,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 73,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 74,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 75,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 76,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 77,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 78,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 79,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 80,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 81,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 82,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 83,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 84,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 85,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 86,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 87,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 88,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 89,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 90,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 91,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 92,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 93,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 94,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 95,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 96,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 97,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 98,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 99,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 100,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 101,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 102,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 103,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 104,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 105,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 106,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 107,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 108,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 109,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 110,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 111,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 112,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 113,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 114,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 115,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 116,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 117,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 118,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 119,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 120,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 121,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 122,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 123,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 124,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 125,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 126,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 127,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 128,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 129,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 130,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 131,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 132,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 133,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 134,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 135,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 136,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 137,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 138,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 139,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 140,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 141,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 142,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 143,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 144,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 145,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 146,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 147,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 148,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 149,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 150,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 151,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 152,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 153,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 154,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 155,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 156,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 157,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 158,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 159,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 160,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 161,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 162,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 163,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 164,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 165,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 166,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 167,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 168,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 169,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 170,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 171,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 172,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 173,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 174,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 175,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 176,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 177,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 178,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 179,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 180,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 181,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 182,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 183,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 184,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 185,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 186,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 187,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 188,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 189,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 190,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 191,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 192,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 193,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 194,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 195,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 196,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 197,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 198,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 199,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 200,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 201,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 202,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 203,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 204,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 205,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 206,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 207,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 208,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 209,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 210,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 211,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 212,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 213,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 214,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 215,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 216,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 217,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 218,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 219,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 220,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 221,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 222,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 223,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 224,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 225,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 226,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 227,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 228,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 229,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 230,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 231,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 232,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 233,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 234,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 235,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 236,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 237,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 238,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 239,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 240,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 241,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 242,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 243,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 244,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 245,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 246,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 247,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 248,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 249,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 250,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 251,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 252,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 253,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 254,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 255,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 256,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 257,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 258,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 259,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 260,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 261,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 262,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 263,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 264,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 265,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 266,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 267,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 268,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 269,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 270,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 271,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 272,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 273,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 274,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 275,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 276,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 277,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 278,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 279,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 280,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 281,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 282,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 283,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 284,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 285,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 286,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 287,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 288,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 289,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 290,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 291,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 292,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 293,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 294,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 295,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 296,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 297,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 298,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 299,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 300,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 301,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 302,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 303,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 304,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 305,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 306,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 307,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 308,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 309,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 310,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 311,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 312,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 313,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 314,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 315,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 316,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 317,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 318,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 319,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 320,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 321,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 322,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 323,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 324,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 325,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 326,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 327,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 328,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 329,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 330,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 331,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 332,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 333,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 334,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 335,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 336,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 337,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 338,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 339,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 340,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 341,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 342,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 343,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 344,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 345,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 346,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 347,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 348,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 349,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 350,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 351,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 352,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 353,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 354,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 355,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 356,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 357,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 358,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 359,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 360,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 361,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 362,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 363,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 364,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 365,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 366,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 367,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 368,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 369,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 370,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 371,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 372,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 373,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 374,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 375,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 376,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 377,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 378,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 379,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 380,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 381,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 382,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 383,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 384,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 385,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 386,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 387,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 388,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 389,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 390,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 391,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 392,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 393,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 394,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 395,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 396,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 397,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 398,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 399,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 400,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 401,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 402,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 403,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 404,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 405,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 406,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 407,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 408,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 409,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 410,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 411,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 412,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 413,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 414,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 415,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 416,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 417,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 418,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 419,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 420,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 421,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 422,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 423,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 424,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 425,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 426,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 427,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 428,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 429,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 430,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 431,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 432,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 433,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 434,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 435,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 436,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 437,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 438,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 439,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 440,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 441,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 442,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 443,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 444,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 445,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 446,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 447,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 448,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 449,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 450,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 451,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 452,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 453,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 454,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 455,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 456,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 457,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 458,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 459,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 460,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 461,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 462,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 463,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 464,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 465,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 466,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 467,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 468,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 469,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 470,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 471,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 472,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 473,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 474,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 475,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 476,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 477,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 478,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 479,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 480,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 481,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 482,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 483,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 484,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 485,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 486,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 487,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 488,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 489,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 490,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 491,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 492,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 493,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 494,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 495,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 496,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 497,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 498,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 499,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 500,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 501,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 502,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 503,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 504,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 505,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 506,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 507,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 508,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 509,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 510,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 511,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 512,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 513,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 514,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 515,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 516,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 517,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 518,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 519,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 520,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 521,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 522,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 523,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 524,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 525,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 526,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 527,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 528,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 529,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 530,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 531,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 532,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 533,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 534,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 535,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 536,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 537,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 538,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 539,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 540,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 541,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 542,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 543,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 544,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 545,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 546,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 547,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 548,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 549,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 550,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 551,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 552,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 553,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 554,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 555,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 556,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 557,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 558,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 559,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 560,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 561,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 562,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 563,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 564,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 565,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 566,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 567,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 568,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 569,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 570,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 571,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 572,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 573,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 574,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 575,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 576,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id": "chapter_1",\n      "source": "../data/chapters/chapter_1.txt"\n    }\n  },\n  {\n    "id": 577,\n    "type": "Relationship",\n    "properties": {"name": "WORKS_FOR"},\n    "source": {\n      "chunk_id": "chunk_5",\n      "chapter_id'

In [ ]:
AIMessage(
    content='After analyzing the given text, I have extracted the relevant entities and relationships in JSON format as follows:\n\n```\n{\n  "nodes": [\n    {\n      "id": "1",\n      "type": "Person",\n      "properties": {\n        "description": "Sherlock Holmes",\n        "context": "A precise, cold and balanced reasoning machine, deeply attracted to the study of crime"\n      }\n    },\n    {\n      "id": "2",\n      "type": "Person",\n      "properties": {\n        "description": "Irene Adler",\n        "context": "The only woman of significance to Holmes, described as late and of dubious memory"\n      }\n    },\n    {\n      "id": "3",\n      "type": "Person",\n      "properties": {\n        "description": "Dr. Watson",\n        "context": "Narrator, former companion of Holmes, medical doctor returned to civil practice"\n      }\n    },\n    {\n      "id": "4",\n      "type": "Location",\n      "properties": {\n        "description": "Baker Street",\n        "context": "Holmes lodgings, also associated with Watson past wooing and dark incidents"\n      }\n    },\n    {\n      "id": "5",\n      "type": "Event",\n      "properties": {\n        "description": "Trepoff Murder",\n        "context": "A murder case in Odessa to which Holmes was summoned"\n      }\n    },\n    {\n      "id": "6",\n      "type": "Event",\n      "properties": {\n        "description": "Tragedy of the Atkinson Brothers",\n        "context": "A singular tragedy at Trincomalee cleared up by Holmes"\n      }\n    },\n    {\n      "id": "7",\n      "type": "Organization",\n      "properties": {\n        "description": "Reigning Family of Holland",\n        "context": "Royal family of Holland for whom Holmes accomplished a delicate mission"\n      }\n    },\n    {\n      "id": "8",\n      "type": "Substance",\n      "properties": {\n        "description": "Cocaine",\n        "context": "A drug used by Holmes, alternating with periods of ambition and fierce energy"\n      }\n    }\n  ],\n  "relationships": [\n    {\n      "source": "3",\n      "target": "1",\n      "type": "COMPANION_OF",\n      "properties": {\n        "status": "estranged due to Watson marriage",\n        "duration": "long-term"\n      }\n    },\n    {\n      "source": "3",\n      "target": "1",\n      "type": "NARRATES_ABOUT",\n      "properties": {\n        "context": "Watson is the first-person narrator describing Holmes and his activities"\n      }\n    },\n    {\n      "source": "1",\n      "target": "2",\n      "type": "ADMIRES",\n      "properties": {\n        "nature": "intellectual admiration, not romantic",\n        "uniqueness": "only woman of significance to Holmes"\n      }\n    },\n    {\n      "source": "3",\n      "target": "4",\n      "type": "VISITS",\n      "properties": {\n        "date": "Twentieth of March 1888"\n      }\n    },\n    {\n      "source": "1",\n      "target": "5",\n      "type": "INVOLVED_IN",\n      "properties": {\n        "context": "Holmes was summoned for the Trepoff murder case in Odessa"\n      }\n    },\n    {\n      "source": "1",\n      "target": "6",\n      "type": "CLEARED_UP",\n      "properties": {\n        "outcome": "successful"\n      }\n    },\n    {\n      "source": "1",\n      "target": "7",\n      "type": "ACCOMPLISHED_MISSION_FOR",\n      "properties": {\n        "context": "Holmes accomplished a delicate mission for the reigning family of Holland"\n      }\n    },\n    {\n      "source": "1",\n      "target": "8",\n      "type": "USES",\n      "properties": {\n        "pattern": "alternating with periods of ambition and fierce energy"\n      }\n    }\n  ]\n}\n```\n\nNote that I have only extracted the entities and relationships mentioned in the given text, and not all possible entities and relationships that could be inferred from the context.',
    additional_kwargs={},
    response_metadata={
        "model": "llama3.1:latest",
        "created_at": "2026-06-16T18:34:58.3856262Z",
        "done": True,
        "done_reason": "stop",
        "total_duration": 18118977400,
        "load_duration": 246143100,
        "prompt_eval_count": 2545,
        "prompt_eval_duration": 2032166000,
        "eval_count": 925,
        "eval_duration": 15330989000,
        "logprobs": None,
        "model_name": "llama3.1:latest",
        "model_provider": "ollama",
    },
    id="lc_run--019ed1b6-d445-78c2-9f16-dc67db19a847-0",
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={"input_tokens": 2545, "output_tokens": 925, "total_tokens": 3470},
)

In [ ]:
AIMessage(
    content='{\n  "nodes": [\n    {\n      "id": "1",\n      "type": "Person",\n      "properties": {\n        "description": "Sherlock Holmes",\n        "context": "Described as the most perfect reasoning and observing machine, cold and precise, abhorrent to softer passions"\n      }\n    },\n    {\n      "id": "2",\n      "type": "Person",\n      "properties": {\n        "description": "Irene Adler",\n        "context": "Referred to as \'the woman\' by Holmes, who eclipses the whole of her sex in his eyes"\n      }\n    },\n    {\n      "id": "3",\n      "type": "Person",\n      "properties": {\n        "description": "Watson (Narrator)",\n        "context": "The speaker observing and describing Holmes\'s attitude towards Irene Adler"\n      }\n    }\n  ],\n  "relationships": [\n    {\n      "source": "1",\n      "target": "2",\n      "type": "ADMires",\n      "properties": {\n        "context": "Holmes considers her \'the woman\' and she eclipses all other women in his eyes, though he feels no emotion akin to love"\n      }\n    },\n    {\n      "source": "3",\n      "target": "1",\n      "type": "OBSERVES",\n      "properties": {\n        "context": "Watson describes Holmes\'s mental state and his unique perspective on Irene Adler"\n      }\n    }\n  ]\n}',
    additional_kwargs={},
    response_metadata={
        "model": "qwen3.6",
        "created_at": "2026-06-16T15:11:16.5170417Z",
        "done": True,
        "done_reason": "stop",
        "total_duration": 190715506200,
        "load_duration": 160099960300,
        "prompt_eval_count": 2211,
        "prompt_eval_duration": 10665399000,
        "eval_count": 359,
        "eval_duration": 19807446000,
        "logprobs": None,
        "model_name": "qwen3.6",
        "model_provider": "ollama",
    },
    id="lc_run--019ed0f9-b496-7d32-81de-18bd7fec41d6-0",
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={"input_tokens": 2211, "output_tokens": 359, "total_tokens": 2570},
)

In [ ]:
AIMessage(
    content='{\n  "nodes": [\n    {\n      "id": "1",\n      "type": "Person",\n      "properties": {\n        "description": "Sherlock Holmes",\n        "context": "The most perfect reasoning and observing machine, cold, precise, balanced mind, abhors emotions akin to love, Bohemian soul, loathes society",\n        "traits": ["logical", "emotionally detached", "bohemian", "observant", "cold"],\n        "habits": ["cocaine use", "studying crime"]\n      }\n    },\n    {\n      "id": "2",\n      "type": "Person",\n      "properties": {\n        "description": "Irene Adler",\n        "context": "The only woman to Holmes, described as \'the\' woman, of dubious and questionable memory",\n        "status": "deceased"\n      }\n    },\n    {\n      "id": "3",\n      "type": "Person",\n      "properties": {\n        "description": "Dr. Watson",\n        "context": "Narrator, recently married, home-centered interests, master of his own establishment",\n        "status": "married"\n      }\n    },\n    {\n      "id": "4",\n      "type": "Organization",\n      "properties": {\n        "description": "Official Police",\n        "context": "Abandoned certain mysteries as hopeless, which Holmes then cleared up"\n      }\n    },\n    {\n      "id": "5",\n      "type": "Location",\n      "properties": {\n        "description": "Baker Street",\n        "context": "Lodgings of Holmes and Watson, where Holmes remained buried among old books"\n      }\n    },\n    {\n      "id": "6",\n      "type": "Location",\n      "properties": {\n        "description": "Odessa",\n        "context": "City to which Holmes was summoned for the Trepoff murder case"\n      }\n    },\n    {\n      "id": "7",\n      "type": "Location",\n      "properties": {\n        "description": "Trincomalee",\n        "context": "Location of the singular tragedy of the Atkinson brothers"\n      }\n    },\n    {\n      "id": "8",\n      "type": "Location",\n      "properties": {\n        "description": "Holland",\n        "context": "Country for whose reigning family Holmes accomplished a delicate mission"\n      }\n    },\n    {\n      "id": "9",\n      "type": "Event",\n      "properties": {\n        "description": "Trepoff Murder",\n        "context": "A murder case in Odessa that summoned Holmes"\n      }\n    },\n    {\n      "id": "10",\n      "type": "Event",\n      "properties": {\n        "description": "Tragedy of the Atkinson Brothers",\n        "context": "A singular tragedy at Trincomalee cleared up by Holmes"\n      }\n    },\n    {\n      "id": "11",\n      "type": "Event",\n      "properties": {\n        "description": "Mission for the Reigning Family of Holland",\n        "context": "A delicate and successful mission accomplished by Holmes"\n      }\n    },\n    {\n      "id": "12",\n      "type": "Substance",\n      "properties": {\n        "description": "Cocaine",\n        "context": "Drug used by Holmes, alternating with ambition"\n      }\n    }\n  ],\n  "relationships": [\n    {\n      "source": "1",\n      "target": "2",\n      "type": "ADMires",\n      "properties": {\n        "nature": "intellectual admiration, not romantic; she eclipses the whole of her sex in his eyes"\n      }\n    },\n    {\n      "source": "3",\n      "target": "1",\n      "type": "COMPANION_OF",\n      "properties": {\n        "status": "estranged due to Watson\'s marriage",\n        "context": "Watson\'s home-centered interests drifted them away"\n      }\n    },\n    {\n      "source": "3",\n      "target": "1",\n      "type": "NARRATES_ABOUT",\n      "properties": {\n        "context": "Watson is the narrator describing Holmes\' nature and activities"\n      }\n    },\n    {\n      "source": "1",\n      "target": "5",\n      "type": "RESIDES_AT",\n      "properties": {\n        "context": "Holmes remained in their lodgings in Baker Street"\n      }\n    },\n    {\n      "source": "1",\n      "target": "12",\n      "type": "USES",\n      "properties": {\n        "pattern": "alternating from week to week between cocaine and ambition"\n      }\n    },\n    {\n      "source": "1",\n      "target": "9",\n      "type": "INVESTIGATED",\n      "properties": {\n        "location": "Odessa",\n        "outcome": "summoned to the case"\n      }\n    },\n    {\n      "source": "1",\n      "target": "10",\n      "type": "SOLVED",\n      "properties": {\n        "location": "Trincomalee",\n        "description": "cleared up the singular tragedy"\n      }\n    },\n    {\n      "source": "1",\n      "target":',
    additional_kwargs={},
    response_metadata={
        "model": "qwen3.6",
        "created_at": "2026-06-16T15:21:34.7380099Z",
        "done": True,
        "done_reason": "length",
        "total_duration": 440880452400,
        "load_duration": 358962277600,
        "prompt_eval_count": 2811,
        "prompt_eval_duration": 10921434000,
        "eval_count": 1285,
        "eval_duration": 70814698000,
        "logprobs": None,
        "model_name": "qwen3.6",
        "model_provider": "ollama",
    },
    id="lc_run--019ed0ff-518d-7e03-b2be-d9ac33319939-0",
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={"input_tokens": 2811, "output_tokens": 1285, "total_tokens": 4096},
)

In [30]:
nodes = []
relations = []
for doc in graph_documents:
    for node in doc.nodes:
        nodes.append(node)
    
    for relation in doc.relationships:
        relations.append(relation)

print(nodes)
print(relations)


[Node(id='Bohemia', type='Location', properties={}), Node(id='A Scandal in Bohemia', type='Work', properties={}), Node(id='Sherlock Holmes', type='Person', properties={}), Node(id='Irene Adler', type='Person', properties={}), Node(id='the trained reasoner', type='Person', properties={}), Node(id='Irene Adler', type='Person', properties={}), Node(id='the observer’s thoughts', type='Work', properties={}), Node(id='cocaine', type='Substance', properties={}), Node(id='Baker Street', type='Location', properties={}), Node(id='Holmes', type='Person', properties={}), Node(id='Trincomalee', type='Location', properties={}), Node(id='reigning family of Holland', type='Organization', properties={}), Node(id='Holland', type='Location', properties={}), Node(id='Odessa', type='Location', properties={}), Node(id='Atkinson brothers', type='Person', properties={}), Node(id='Trepoff murder', type='Event', properties={})]
[Relationship(source=Node(id='Bohemia', type='Location', properties={}), target=Node

In [229]:
import json
from typing import List, Dict, Any

from pydantic import BaseModel, Field, ValidationError
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_community.graphs.graph_document import Node, Relationship, GraphDocument

In [ ]:


def cast_to_graph_document(json_object:any):
    nodes: list[Node] = []
    for node in json_object["nodes"]:
        nodes.append(serialize_node(node))

    relationships: list[Relationship] = []
    for rel in json_object["relationships"]:
        relationships.append(serialize_relationship(rel, nodes))
    
    source: Document = Document(metadata=json_object["document"]["metadata"], page_content=json_object["document"]["page_content"])
    return GraphDocument(nodes=nodes, relationships=relationships, source=source)


def convert_to_graph_doc(s: str):
    graph_documents = eval(s, {
        "GraphDocument": GraphDocument,
        "Document": Document,
        "Node": Node,
        "Relationship": Relationship,
    })
    return graph_documents

In [231]:
class NodeModel(BaseModel):
    id: str
    type: str
    properties: Dict[str, Any] = Field(default_factory=dict)


class RelationshipModel(BaseModel):
    source: NodeModel
    target: NodeModel
    type: str
    properties: Dict[str, Any] = Field(default_factory=dict)


class DocumentModel(BaseModel):
    metadata: Dict[str, Any] = Field(default_factory=dict)
    page_content: str


class ExtractionResult(BaseModel):
    nodes: List[NodeModel] = Field(default_factory=list)
    relationships: List[RelationshipModel] = Field(default_factory=list)
    document: DocumentModel

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a high-precision information extraction system designed to construct knowledge graphs from a single source document.

Task
- Extract entities and relationships from the given source document and return a single JSON object containing:
  - nodes: list of node objects extracted from this document
  - relationships: list of relationship objects extracted from this document
  - document: the source document metadata and page_content

Hard rules
- Output only valid JSON.
- Do not include any text outside the JSON object.
- Do not return Python code, GraphDocument(...), markdown, explanations, or commentary.
- Process the input document as a single unit and extract only facts explicitly present in that document.
- Do not invent facts or infer unstated information.
- Return empty lists for nodes or relationships when the document contains none.

Canonicalization & coreference
- Resolve all coreferences, including pronouns, partial names, epithets, aliases, and descriptions, to the single canonical node ID that is the most complete explicit name present in the same document.
- Examples:
  - map "Holmes" or "he" to "Sherlock Holmes" if "Sherlock Holmes" appears in the same document
  - map "Watson" to "Dr. Watson" if "Dr. Watson" appears in the same document
  - map "the woman" to "Irene Adler" if "Irene Adler" appears in the same document
- If the full canonical name does not appear in the document and the reference is ambiguous, keep the original mention only if it is explicitly present in the text and classify it as Alias when appropriate.
- After resolution, ensure each real-world entity appears exactly once in nodes.
- Relationship source and target must reference canonical node objects, not pronouns like "he", "she", "you", "I", or shortened mentions like "Holmes" when a fuller canonical name exists in the same document.

Node object format
- id: canonical, human-readable name exactly as found in the document
- type: general, simple node type
- properties: object with only explicit properties from the document; otherwise empty object

Relationship object format
- source: node object
- target: node object
- type: general, timeless relationship type
- properties: object with only explicit relationship context from the document; otherwise empty object

Document object format
- document must contain metadata and page_content from the input source document.

Output structure
Return a single JSON object with exactly these keys:
- nodes
- relationships
- document
"""
    ),
    (
        "human",
        "Input document:\n{input_document}"
    )
])


llm_json = ChatOllama(
    model="qwen3.6:latest",
    base_url="http://192.168.178.66:11434",
    temperature=0,
    format=ExtractionResult.model_json_schema(),
    reasoning=False,
    keep_alive="24h",
    num_predict=-1,
)

# llm_json = llm.with_structured_output(ExtractionResult, include_raw=True)

# llm = ChatOllama(
#     model="qwen3.6:latest",
#     base_url="http://192.168.178.66:11434",
#     temperature=0,
#     reasoning=False,
#     keep_alive="24h",
#     format="json",
#     num_ctx=32768,       # context window: how much input+output the model can "see"
#     num_predict=-1,     # output token limit: -1 = no limit (generate until done)
# )
# llm.invoke("Say hello in one sentence.")


json_chain = prompt | llm_json

# response = json_chain.invoke({"input_document":json.dumps(
#                 {
#                     "metadata": documents[2].metadata,
#                     "page_content": documents[2].page_content,
#                 },
#                 ensure_ascii=False,
#             )})

In [297]:
response = json_chain.invoke(
        {
            "input_document": documents[1]
        }
    )
response

AIMessage(content='{\n  "nodes": [\n    {\n      "id": "Sherlock Holmes",\n      "type": "Person",\n      "properties": {}\n    },\n    {\n      "id": "Baker Street",\n      "type": "Location",\n      "properties": {}\n    },\n    {\n      "id": "Odessa",\n      "type": "Location",\n      "properties": {}\n    },\n    {\n      "id": "Trincomalee",\n      "type": "Location",\n      "properties": {}\n    },\n    {\n      "id": "Holland",\n      "type": "Location",\n      "properties": {}\n    },\n    {\n      "id": "Trepoff murder",\n      "type": "Crime",\n      "properties": {}\n    },\n    {\n      "id": "Atkinson brothers",\n      "type": "Person",\n      "properties": {}\n    },\n    {\n      "id": "singular tragedy of the Atkinson brothers",\n      "type": "Event",\n      "properties": {}\n    },\n    {\n      "id": "reigning family of Holland",\n      "type": "Person",\n      "properties": {}\n    },\n    {\n      "id": "official police",\n      "type": "Organisation",\n      "pro

In [356]:
# def validate_node_type(node_type: str) -> str:
#     return node_type if node_type in ALLOWED_NODES else "Alias"


# def validate_relationship_type(rel_type: str) -> str:
#     return rel_type if rel_type in ALLOWED_RELATIONSHIPS else rel_type.upper().strip()


# def extraction_to_graph_document(result: ExtractionResult) -> GraphDocument:
#     all_nodes = {}
#     for n in result.nodes:
#         node = Node(
#             id=n.id,
#             type=validate_node_type(n.type),
#             properties=n.properties or {},
#         )
#         dedup_nodes[(node.id, node.type)] = node

#     node_by_id = {}
#     for node in dedup_nodes.values():
#         node_by_id[node.id] = node

#     relationships = []
#     for r in result.relationships:
#         source_id = r.source.id
#         target_id = r.target.id

#         if source_id not in node_by_id:
#             node_by_id[source_id] = Node(
#                 id=source_id,
#                 type=validate_node_type(r.source.type),
#                 properties=r.source.properties or {},
#             )

#         if target_id not in node_by_id:
#             node_by_id[target_id] = Node(
#                 id=target_id,
#                 type=validate_node_type(r.target.type),
#                 properties=r.target.properties or {},
#             )

#         relationships.append(
#             Relationship(
#                 source=node_by_id[source_id],
#                 target=node_by_id[target_id],
#                 type=validate_relationship_type(r.type),
#                 properties=r.properties or {},
#             )
#         )

#     source_doc = Document(
#         page_content=result.document.page_content,
#         metadata=result.document.metadata,
#     )

#     return GraphDocument(
#         nodes=list(node_by_id.values()),
#         relationships=relationships,
#         source=source_doc,
#     )

def extract_graph_document(resolved: ExtractionResult):
    return GraphDocument(nodes=resolved.nodes, relationships=resolved.relationships, source=resolved.document)

def resolve_results(validated_result: GraphDocument):
    resolved_nodes = []
    resolved_relationships = []
    alias_map = {
        "holmes": "Sherlock Holmes",
        "sherlock": "Sherlock Holmes",
        "watson": "Dr. Watson",
        "the woman": "Irene Adler",
        "miss adler": "Irene Adler",
        "mr. holmes": "Sherlock Holmes",
        "i": "Dr. Watson",
        "narrator": "Dr. Watson",
        "he": "Sherlock Holmes"
    }
    for node in validated_result.nodes:
        new_node = deepcopy(node)
        if node.id.lower() in alias_map:
            new_node.id = alias_map[node.id.lower()]
            resolved_nodes.append(new_node)
        resolved_nodes.append(new_node)

    for rel in validated_result.relationships:
        source_node = rel.source
        target_node = rel.target
        new_rel = deepcopy(rel)
        if source_node.id.lower() in alias_map:
            new_node = deepcopy(source_node)
            new_node.id = alias_map[source_node.id.lower()]
            new_rel.source = new_node
        if target_node.id.lower() in alias_map:
            new_node = deepcopy(target_node)
            new_node.id = alias_map[target_node.id.lower()]
            new_rel.target = new_node
        
        resolved_relationships.append(new_rel)
    
    return GraphDocument(nodes=resolved_nodes, relationships=resolved_relationships, source=validated_result.source)


def extract_graph_document_json_enforced(doc: Document) -> GraphDocument:
    response = json_chain.invoke({"input_document": doc})
    json_data = json.loads(response.content)

    validated = ExtractionResult.model_validate(json_data)
    graph_doc = extract_graph_document(validated)
    return resolve_results(graph_doc)


def process_documents_json_enforced(documents: List[Document]) -> List[GraphDocument]:
    graph_docs = []
    for i, doc in enumerate(documents):
        try:
            graph_doc = extract_graph_document_json_enforced(doc)
            graph_docs.append(graph_doc)
            print(f"Processed document {i+1}/{len(documents)}")
        except (ValidationError, json.JSONDecodeError, Exception) as e:
            print(f"Failed document {i+1}: {e}")
    return graph_docs





In [305]:
import re

def normalize_name(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r"[^\w\s]", "", text) # removing special characters
    text = re.sub(r"\s+", " ", text) # collapsing whitespaces
    return text





In [ ]:
alias_map = {
    "holmes": "Sherlock Holmes",
    "sherlock": "Sherlock Holmes",
    "watson": "Dr. Watson",
    "the woman": "Irene Adler",
    "miss adler": "Irene Adler",
}

In [13]:
# graph_docs = []
# for doc in documents[:5]:

directory = "../data/chapters/"
for file in os.listdir(directory):
    print(f"processing {file}...")
    loader = TextLoader(file_path=f"../data/chapters/{file}")
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
    documents = text_splitter.split_documents(documents=docs)

    graph_docs = process_documents_json_enforced(documents)

    with open(f"../data/local/{file}", "w") as file_out:
        file_out.write(str(graph_docs))

processing chapter_8.txt...


NameError: name 'process_documents_json_enforced' is not defined

In [347]:
graph_docs

[GraphDocument(nodes=[Node(id='Sherlock Holmes', type='Person', properties={}), Node(id='Lord St. Simon', type='Person', properties={}), Node(id='Baker Street', type='Location', properties={}), Node(id='Afghan campaign', type='Event', properties={}), Node(id='jezail bullet', type='Object', properties={}), Node(id='THE ADVENTURE OF THE NOBLE BACHELOR', type='Case', properties={})], relationships=[Relationship(source=Node(id='Sherlock Holmes', type='Person', properties={}), target=Node(id='THE ADVENTURE OF THE NOBLE BACHELOR', type='Case', properties={}), type='INVESTIGATED', properties={}), Relationship(source=Node(id='Sherlock Holmes', type='Person', properties={}), target=Node(id='Baker Street', type='Location', properties={}), type='RESIDED_AT', properties={})], source=Document(metadata={'source': '../data/chapters/chapter_10.txt'}, page_content='X. THE ADVENTURE OF THE NOBLE BACHELOR\n\n\nThe Lord St. Simon marriage, and its curious termination, have long\nceased to be a subject of 

In [359]:
directory = "../data/local/"
for file in os.listdir(directory):
    with open(f"{directory}{file}", "r", encoding="utf-8") as file_in:
        string_graph_docs = file_in.read()
    parsed_graph_document = convert_to_graph_doc(string_graph_docs)
    resolved_graph_docs = []
    for graph_doc in parsed_graph_document:
        resolved_graph_docs.append(resolve_results(graph_doc))
        print(f"resolved {len(resolved_graph_docs)}/{len(parsed_graph_document)}")
    graph.add_graph_documents(
        resolved_graph_docs,
        baseEntityLabel=True,
        include_source=True,
    )
    print(f"Added {file} to graph.")

resolved 1/44
resolved 2/44
resolved 3/44
resolved 4/44
resolved 5/44
resolved 6/44
resolved 7/44
resolved 8/44
resolved 9/44
resolved 10/44
resolved 11/44
resolved 12/44
resolved 13/44
resolved 14/44
resolved 15/44
resolved 16/44
resolved 17/44
resolved 18/44
resolved 19/44
resolved 20/44
resolved 21/44
resolved 22/44
resolved 23/44
resolved 24/44
resolved 25/44
resolved 26/44
resolved 27/44
resolved 28/44
resolved 29/44
resolved 30/44
resolved 31/44
resolved 32/44
resolved 33/44
resolved 34/44
resolved 35/44
resolved 36/44
resolved 37/44
resolved 38/44
resolved 39/44
resolved 40/44
resolved 41/44
resolved 42/44
resolved 43/44
resolved 44/44
Added chapter_8.txt to graph.
resolved 1/43
resolved 2/43
resolved 3/43
resolved 4/43
resolved 5/43
resolved 6/43
resolved 7/43
resolved 8/43
resolved 9/43
resolved 10/43
resolved 11/43
resolved 12/43
resolved 13/43
resolved 14/43
resolved 15/43
resolved 16/43
resolved 17/43
resolved 18/43
resolved 19/43
resolved 20/43
resolved 21/43
resolved 22/4

In [357]:
clean_graph()
# graph.add_graph_documents(
#     parsed_graph_document,
#     baseEntityLabel=True,
#     include_source=True,
# )

In [360]:
embeddings = OllamaEmbeddings(
    model="mxbai-embed-large",
)

vector_index = Neo4jVector.from_existing_graph(
    embeddings,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)
vector_retriever = vector_index.as_retriever()

In [1]:
llm = ChatOllama(
    model="qwen3.6:latest",
    base_url="http://192.168.178.66:11434",
    temperature=0,
    reasoning=False,
    keep_alive="24h",
    # format="json"
)

llm.invoke("Say hello in one sentence!")

NameError: name 'ChatOllama' is not defined

In [ ]:
driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USERNAME"],
                os.environ["NEO4J_PASSWORD"]))

def create_fulltext_index(tx):
    query = '''
    CREATE FULLTEXT INDEX `fulltext_entity_id` 
    FOR (n:__Entity__) 
    ON EACH [n.id];
    '''
    tx.run(query)

# Function to execute the query
def create_index():
    with driver.session(database="local") as session:
        session.execute_write(create_fulltext_index)
        print("Fulltext index created successfully.")

# Call the function to create the index
try:
    create_index()
except:
    print("Index creation failed")
    pass

# Close the driver connection
driver.close()

Fulltext index created successfully.


In [365]:

class Entities(BaseModel):
    """Identifying information about entities."""

    names: list[str] = Field(
        ...,
        description="All the person, organization, or business entities that "
        "appear in the text",
    )

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting organization and person entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following "
            "input: {question}",
        ),
    ]
)


entity_chain = prompt | llm.with_structured_output(Entities, method="function_calling")

In [366]:
entity_chain.invoke("Who are Sherlock Holmes and Irene Adler?")

Entities(names=['Sherlock Holmes', 'Irene Adler'])

In [ ]:
print(graph_retriever("Sherlock Holmes"))

Irene Adler - RESIDES_AT -> Serpentine-mews
Irene Adler - MARRIED_TO -> Norton
Irene Adler - POSSESSES -> photograph
Irene Adler - THREATENS_TO_SEND_PHOTOGRAPH_TO -> Your Majesty
Irene Adler - MARRIED -> Godfrey Norton
Irene Adler - BIRTH_PLACE -> New Jersey
Irene Adler - AFFILIATION -> La Scala
Irene Adler - EMPLOYMENT -> Imperial Opera of Warsaw
Irene Adler - RESIDENCE -> London
Irene Adler - WILL_SEND_PHOTOGRAPH_ON_DAY_OF -> betrothal
Irene Adler - LOCATEDAT -> Briony Lodge
Irene Adler - TRAVELING_WITH -> Norton
Irene Adler - LEAVING -> England
Irene Adler - ADDRESSED_TO -> Sherlock Holmes, Esq.
Irene Adler - REFERS_TO -> Irene Adler
Irene Adler - ADVISES -> Dr. Watson
Irene Adler - WARNS -> Dr. Watson
Sherlock Holmes - IS_ASSOCIATED_WITH -> Irene Adler
clergyman - OFFICIATED_MARRIAGE_FOR -> Irene Adler
strange visitor - ACQUAINTANCE -> Irene Adler
Sherlock Holmes - BIOGRAPHY_SOURCE -> Irene Adler
strange visitor - WROTE_LETTERS_TO -> Irene Adler
Sherlock Holmes - PROTECTED -> Irene

In [369]:
vector_data = [el.page_content for el in vector_retriever.invoke("Who is Irene Adler?")]

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

In [370]:
def full_retriever(question: str):
    graph_data = graph_retriever(question)
    vector_data = [el.page_content for el in vector_retriever.invoke(question)]
    final_data = f"""Graph data:
{graph_data}
vector data:
{"#Document ". join(vector_data)}
    """
    return final_data

In [371]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
        {
            "context": full_retriever,
            "question": RunnablePassthrough(),
        }
    | prompt
    | llm
    | StrOutputParser()
)

In [372]:
chain.invoke(input="How are Irene Adler and Sherlock Holmes related?")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

"Based on the provided context, Sherlock Holmes respects, protects, and is associated with Irene Adler, while also seeking her papers and photograph. Additionally, Irene Adler's biography serves as a source for Sherlock Holmes's biography."